# 2D MT: L2 vs 6D OT under non-Gaussian noise

**Description**  
This notebook compares **L2 (MSE)** and **6D OT (Sinkhorn)** data loss for 2D MT inversion on the **same simple model** and the **same synthetic dataset** with **non-Gaussian noise** (Gaussian + random outliers). Model parameters (grid, frequencies, stations, true conductivity) match **example_2d.ipynb**.

- **True model**: 0.01 S/m half-space, one conductive block (1 S/m) in the center; grid and layout as in example_2d.
- **Data**: One synthetic dataset with `noise_type="nongaussian"` (and optional `outlier_frac` / `outlier_strength`), shared by both inversions via the same `random_seed`.
- **Runs**: First inversion with `mode="mse"` (L2), second with `mode="6dot"` (6D OT); both use the same regularization and similar settings where applicable.
- **Output**: Side-by-side resistivity sections (true vs L2 vs 6D OT) and loss/misfit curves for comparison.

In [ ]:
import sys
sys.path.insert(0, '..')
import torch
import numpy as np

from src.mt2d_inv import MT2DInverter

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# ---------- Model parameters (same as example_2d.ipynb) ----------
freqs = torch.logspace(0.5, -4, 20)
stations = torch.linspace(-15000, 15000, 31)

nza = 10
z_air = -np.logspace(np.log10(10), np.log10(50000), nza)
z_air = np.flip(z_air)
z_air = np.append(z_air, 0)
z_shallow = np.logspace(np.log10(100), np.log10(5000), 10)
z_deep = np.logspace(np.log10(6000), np.log10(100000), 20)
zn = np.concatenate([z_air[:-1], np.array([0]), np.concatenate([z_shallow, z_deep])])

y_center = np.linspace(-10000, 10000, 21)
y_left = np.flip(-np.logspace(np.log10(11000), np.log10(50000), 10))
y_right = np.logspace(np.log10(11000), np.log10(50000), 10)
yn = np.concatenate([y_left, y_center, y_right])

yc = (yn[:-1] + yn[1:]) / 2.0
zc = (zn[:-1] + zn[1:]) / 2.0
Y, Z = np.meshgrid(yc, zc)
sig_true = np.ones_like(Y) * 0.01
sig_true[Z < 0] = 1e-9
mask_anomaly = (np.abs(Y) <= 3000) & (Z >= 1000) & (Z <= 10000)
sig_true[mask_anomaly] = 1.0
sig_true = torch.tensor(sig_true, dtype=torch.float64, device=device)
yn = torch.tensor(yn, dtype=torch.float64)
zn = torch.tensor(zn, dtype=torch.float64)

print(f"Grid: {len(zn)-1} x {len(yn)-1} cells; depth to {zn[-1]/1e3:.1f} km")

## 1. L2 (MSE) inversion

Same data generated with non-Gaussian noise; inversion with `mode="mse"`.

In [ ]:
inv_l2 = MT2DInverter(
    yn=yn, zn=zn, freqs=freqs, stations=stations,
    device=device, random_seed=123
)
inv_l2.set_forward_operator(nza=nza)
inv_l2.sig_true = sig_true
inv_l2.create_synthetic_data(
    noise_level=0.01,
    noise_type="nongaussian",
    outlier_frac=0.05,
    outlier_strength=4.0
)
inv_l2.initialize_model(initial_sigma=0.01)

lam0_l2 = inv_l2.compute_initial_lambda(mode="mse", use_ot_weights=False)
final_sigma_l2 = inv_l2.run_inversion(
    n_epochs=200,
    mode="mse",
    current_lambda=lam0_l2,
    use_adaptive_lambda=True,
    use_ot_weights=False,
    verbose=True,
    progress_interval=25,
)
print("L2 inversion done. Final conductivity range:", final_sigma_l2.min().item(), "-", final_sigma_l2.max().item())

In [ ]:
inv_l2.plot_model_comparison(use_resistivity=True, cmap="jet_r", xlim=[-20, 20], ylim=[50, 0])
inv_l2.plot_loss_history(target_misfit=1.0)

## 2. 6D OT (Sinkhorn) inversion

Same model and same non-Gaussian data (same seed 123); inversion with `mode="6dot"`.

In [ ]:
inv_ot = MT2DInverter(
    yn=yn, zn=zn, freqs=freqs, stations=stations,
    device=device, random_seed=123
)
inv_ot.set_forward_operator(nza=nza)
inv_ot.sig_true = sig_true
inv_ot.create_synthetic_data(
    noise_level=0.01,
    noise_type="nongaussian",
    outlier_frac=0.05,
    outlier_strength=4.0
)
inv_ot.initialize_model(initial_sigma=0.01)

lam0_ot = inv_ot.compute_initial_lambda(mode="6dot", use_ot_weights=True)
final_sigma_ot = inv_ot.run_inversion(
    n_epochs=200,
    mode="6dot",
    current_lambda=lam0_ot,
    use_adaptive_lambda=True,
    use_ot_weights=True,
    verbose=True,
    progress_interval=25,
)
print("6D OT inversion done. Final conductivity range:", final_sigma_ot.min().item(), "-", final_sigma_ot.max().item())

In [ ]:
inv_ot.plot_model_comparison(use_resistivity=True, cmap="jet_r", xlim=[-20, 20], ylim=[50, 0])
inv_ot.plot_loss_history(target_misfit=1.0)